# Hệ Thống Giám Sát Hành Vi Gian Lận Của Thí Sinh Trong Phòng Thi Bằng AI (YOLO26s + YOLO26s-Pose)
**Bám sát Đề cương Tiểu luận**:
1. **4 lớp nhận diện vật thể (`Detection`)**: `student` (0), `phone` (1), `paper` (2), `calculator` (3).
2. **11 điểm khớp thân trên (`Upper-Body Pose Keypoints`) của `student`**: `nose`, `left_eye`, `right_eye`, `left_ear`, `right_ear`, `left_shoulder`, `right_shoulder`, `left_elbow`, `right_elbow`, `left_wrist`, `right_wrist` *(đã loại bỏ 6 điểm thân dưới bị bàn che khuất)*.
3. **Thư mục lưu kết quả trên Google Drive**: Toàn bộ checkpoint (`best.pt`, `last.pt`, `last_backup.pt`), bảng thống kê CSV, biểu đồ và ảnh cảnh báo đều được lưu tự động vào **`/content/drive/MyDrive/ket_thuc_hoc_phan_AI`**.

In [ ]:
!nvidia-smi
!pip -q install -U ultralytics pycocotools pandas opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Cấu hình Đường dẫn & Giải nén Dữ liệu (`/content/drive/MyDrive/dataset_sv_vi_pham_yolo.zip`)
- Đọc file `/content/drive/MyDrive/dataset_sv_vi_pham_yolo.zip` bạn đã tải lên gốc Google Drive.
- Tự động tạo thư mục `/content/drive/MyDrive/ket_thuc_hoc_phan_AI` trên Drive để lưu toàn bộ kết quả và checkpoint.

In [ ]:
import os, glob, json, shutil, zipfile, yaml, torch

# 1. Đường dẫn chuẩn trên Google Drive
ZIP_YOLO         = '/content/drive/MyDrive/dataset_sv_vi_pham_yolo.zip'
ZIP_LABELS_PATCH = '/content/drive/MyDrive/labels_student_pose_11kpts.zip'  # Bản cập nhật nhãn 3.5MB (nếu bạn tải bản zip 928MB lúc 09:32)
OUT_DIR          = '/content/drive/MyDrive/ket_thuc_hoc_phan_AI'            # Thư mục lưu toàn bộ kết quả trên Google Drive
os.makedirs(OUT_DIR, exist_ok=True)

BASE_DIR       = '/content/dataset_sv_vi_pham'
DET_DIR        = f'{BASE_DIR}/det'
POSE_DIR       = f'{BASE_DIR}/pose'
DATA_DET_YAML  = f'{DET_DIR}/data.yaml'
DATA_POSE_YAML = f'{POSE_DIR}/data_pose.yaml'

DET_CLASSES = ['student', 'phone', 'paper', 'calculator']
UPPER_KPTS  = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist'
]
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9]

if not os.path.exists(DATA_DET_YAML):
    assert os.path.exists(ZIP_YOLO), f'Không tìm thấy file dataset tại: {ZIP_YOLO}. Hãy kiểm tra trên Drive của tôi!'
    raw_extract = f'{BASE_DIR}/raw_unzip'
    os.makedirs(raw_extract, exist_ok=True)
    print(f'Đang giải nén {ZIP_YOLO} ra ổ cứng Colab...')
    with zipfile.ZipFile(ZIP_YOLO, 'r') as z:
        z.extractall(raw_extract)

    # Nếu có file patch nhãn 3.5MB (labels_student_pose_11kpts.zip) trên Drive thì ghi đè nhãn mới nhất
    if os.path.exists(ZIP_LABELS_PATCH):
        print('Đang cập nhật bộ nhãn 4 lớp (student, phone, paper, calculator) + 11 điểm Pose thân trên từ labels_student_pose_11kpts.zip...')
        patch_dir = f'{BASE_DIR}/patch_lbls'
        with zipfile.ZipFile(ZIP_LABELS_PATCH, 'r') as z:
            z.extractall(patch_dir)
        # Thu thập toàn bộ đường dẫn ảnh theo tên file (stem)
        all_imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{raw_extract}/*/images/*.jpg')}
        for sp in ('train', 'valid', 'test'):
            os.makedirs(f'{DET_DIR}/{sp}/images', exist_ok=True)
            os.makedirs(f'{DET_DIR}/{sp}/labels', exist_ok=True)
            os.makedirs(f'{POSE_DIR}/{sp}/images', exist_ok=True)
            os.makedirs(f'{POSE_DIR}/{sp}/labels', exist_ok=True)
            for lbl_f in glob.glob(f'{patch_dir}/{sp}/labels/*.txt'):
                stem = os.path.splitext(os.path.basename(lbl_f))[0]
                if stem not in all_imgs: continue
                det_img  = f'{DET_DIR}/{sp}/images/{stem}.jpg'
                pose_img = f'{POSE_DIR}/{sp}/images/{stem}.jpg'
                shutil.move(all_imgs[stem], det_img)
                try: os.link(det_img, pose_img)
                except Exception: shutil.copy2(det_img, pose_img)
                shutil.move(lbl_f, f'{DET_DIR}/{sp}/labels/{stem}.txt')
                pose_f = f'{patch_dir}/{sp}/labels_pose/{stem}.txt'
                if os.path.exists(pose_f):
                    shutil.move(pose_f, f'{POSE_DIR}/{sp}/labels/{stem}.txt')
        shutil.rmtree(patch_dir, ignore_errors=True)
    else:
        # Kiểm tra xem bản zip có sẵn labels_pose hay là bản 6 lớp trước đó
        has_pose = os.path.exists(f'{raw_extract}/train/labels_pose')
        old_yaml = yaml.safe_load(open(f'{raw_extract}/data.yaml', encoding='utf-8')) if os.path.exists(f'{raw_extract}/data.yaml') else {}
        is_old_6cls = (old_yaml.get('nc') == 6)
        # Ánh xạ 6 lớp cũ ['person','paper','calculator','laptop','phone','book'] -> 4 lớp ['student','phone','paper','calculator']
        OLD2NEW = {0: 0, 4: 1, 1: 2, 5: 2, 2: 3}
        for sp in ('train', 'valid', 'test'):
            os.makedirs(f'{DET_DIR}/{sp}/images', exist_ok=True)
            os.makedirs(f'{DET_DIR}/{sp}/labels', exist_ok=True)
            os.makedirs(f'{POSE_DIR}/{sp}/images', exist_ok=True)
            os.makedirs(f'{POSE_DIR}/{sp}/labels', exist_ok=True)
            for img_p in glob.glob(f'{raw_extract}/{sp}/images/*.jpg'):
                fn = os.path.basename(img_p); stem = os.path.splitext(fn)[0]
                det_img = f'{DET_DIR}/{sp}/images/{fn}'; pose_img = f'{POSE_DIR}/{sp}/images/{fn}'
                shutil.move(img_p, det_img)
                try: os.link(det_img, pose_img)
                except Exception: shutil.copy2(det_img, pose_img)
                det_lbl = f'{raw_extract}/{sp}/labels/{stem}.txt'
                if os.path.exists(det_lbl):
                    if is_old_6cls:
                        lines = []
                        for l in open(det_lbl):
                            vs = l.split()
                            if vs and int(vs[0]) in OLD2NEW:
                                lines.append(f'{OLD2NEW[int(vs[0])]} ' + ' '.join(vs[1:5]))
                        open(f'{DET_DIR}/{sp}/labels/{stem}.txt', 'w').write('\n'.join(lines) + ('\n' if lines else ''))
                    else:
                        shutil.move(det_lbl, f'{DET_DIR}/{sp}/labels/{stem}.txt')
                pose_lbl = f'{raw_extract}/{sp}/labels_pose/{stem}.txt'
                if has_pose and os.path.exists(pose_lbl):
                    shutil.move(pose_lbl, f'{POSE_DIR}/{sp}/labels/{stem}.txt')
    shutil.rmtree(raw_extract, ignore_errors=True)

with open(DATA_DET_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump({'path': DET_DIR, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 4, 'names': DET_CLASSES}, f, sort_keys=False)

with open(DATA_POSE_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump({'path': POSE_DIR, 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['student'], 'kpt_shape': [11, 3], 'flip_idx': FLIP_IDX}, f, sort_keys=False)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 16
IMGSZ   = 960 if vram_gb >= 20 else 640
BATCH   = 16
EPOCHS, PATIENCE, SEED, WORKERS = 100, 30, 0, 2
print(f'✅ Đã cấu hình xong! Thư mục lưu kết quả trên Drive: {OUT_DIR}')
print(f'GPU VRAM: {vram_gb:.1f} GB -> IMGSZ={IMGSZ}, BATCH={BATCH}')
print(f"Số ảnh Train: {len(glob.glob(f'{DET_DIR}/train/images/*'))} | Valid: {len(glob.glob(f'{DET_DIR}/valid/images/*'))} | Test: {len(glob.glob(f'{DET_DIR}/test/images/*'))}")

## 2. Cơ chế Auto-Backup & Auto-Resume Checkpoint (`ket_thuc_hoc_phan_AI`)
- Sau mỗi epoch, trọng số `last.pt` và bản sao dự phòng `last_backup.pt` được lưu trực tiếp vào `/content/drive/MyDrive/ket_thuc_hoc_phan_AI/<tên_model>/weights/`.
- Khi Colab bị ngắt kết nối giữa chừng, bạn chỉ cần chạy lại từ trên xuống: hàm `train_with_backup` sẽ tự động nạp `last.pt` và train tiếp từ đúng epoch bị ngắt.

In [ ]:
import os, json, time, shutil, torch, pandas as pd
from ultralytics import YOLO

def _valid_ckpt(path):
    if not os.path.exists(path): return None
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f'Cảnh báo: {path} bị lỗi ({e})')
        return None

def _backup_callback(trainer):
    last_pt = str(trainer.last)
    if os.path.exists(last_pt):
        try:
            shutil.copy2(last_pt, last_pt.replace('last.pt', 'last_backup.pt'))
        except Exception:
            pass

def train_with_backup(run_name, base_weights, data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH):
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    run_dir   = f'{OUT_DIR}/{run_name}'
    last_pt   = f'{run_dir}/weights/last.pt'
    backup_pt = f'{run_dir}/weights/last_backup.pt'
    best_pt   = f'{run_dir}/weights/best.pt'
    state_f   = f'{run_dir}/train_state.json'
    state     = json.load(open(state_f)) if os.path.exists(state_f) else {'train_h': 0.0, 'vram_peak_GB': 0.0}

    ckpt = _valid_ckpt(last_pt)
    if ckpt is None and os.path.exists(backup_pt):
        print(f'[{run_name}] Khôi phục last.pt từ bản sao lưu an toàn last_backup.pt...')
        shutil.copy2(backup_pt, last_pt)
        ckpt = _valid_ckpt(last_pt)

    t0 = time.time()
    if ckpt is not None and ckpt.get('epoch', -1) < 0 and os.path.exists(best_pt):
        print(f'✅ [{run_name}] Đã huấn luyện hoàn tất từ phiên trước! Bỏ qua train và chuyển sang đánh giá.')
    elif ckpt is not None and ckpt.get('epoch', -1) >= 0:
        ep_done = ckpt.get('epoch', 0) + 1
        print(f'🔄 [{run_name}] Phát hiện checkpoint tại epoch {ep_done}/{epochs} -> Tự động RESUME huấn luyện tiếp!')
        model = YOLO(last_pt)
        model.add_callback('on_fit_epoch_end', _backup_callback)
        model.train(resume=True)
    else:
        print(f'🚀 [{run_name}] Bắt đầu huấn luyện mới từ {base_weights} (imgsz={imgsz}, batch={batch})...')
        model = YOLO(base_weights)
        model.add_callback('on_fit_epoch_end', _backup_callback)
        model.train(
            data=data_yaml, epochs=epochs, imgsz=imgsz, batch=batch, patience=PATIENCE,
            seed=SEED, workers=WORKERS, project=OUT_DIR, name=run_name, exist_ok=True, plots=True, save=True,
            cos_lr=True, close_mosaic=15, mosaic=1.0, scale=0.6
        )

    train_h = state.get('train_h', 0.0) + (time.time() - t0) / 3600
    cur_vram = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    vram = max(state.get('vram_peak_GB', 0.0), cur_vram)
    os.makedirs(run_dir, exist_ok=True)
    json.dump({'train_h': train_h, 'vram_peak_GB': vram}, open(state_f, 'w'))
    return YOLO(best_pt), round(train_h, 2), round(vram, 2)

## 3. Huấn luyện Mô hình Nhận diện Vật thể tốt nhất (`yolo26s.pt`)
Nhận diện 4 lớp đối tượng phòng thi: `student`, `phone`, `paper`, `calculator`. Kết quả lưu tại `/content/drive/MyDrive/ket_thuc_hoc_phan_AI/yolo26s_det_4classes`.

In [ ]:
best_det, det_h, det_vram = train_with_backup(
    run_name='yolo26s_det_4classes',
    base_weights='yolo26s.pt',
    data_yaml=DATA_DET_YAML
)

m_val  = best_det.val(data=DATA_DET_YAML, imgsz=IMGSZ, batch=BATCH, split='val',  project=OUT_DIR, name='yolo26s_det_val',  exist_ok=True)
m_test = best_det.val(data=DATA_DET_YAML, imgsz=IMGSZ, batch=BATCH, split='test', project=OUT_DIR, name='yolo26s_det_test', exist_ok=True)

det_summary = {
    'model': 'yolo26s_det_4classes',
    'val_mAP50': round(float(m_val.box.map50), 4),
    'val_mAP50-95': round(float(m_val.box.map), 4),
    'test_mAP50': round(float(m_test.box.map50), 4),
    'test_mAP50-95': round(float(m_test.box.map), 4),
    'Precision': round(float(m_test.box.mp), 4),
    'Recall': round(float(m_test.box.mr), 4),
    'infer_ms': round(float(m_test.speed['inference']), 2),
    'vram_GB': det_vram,
    'train_h': det_h,
}
for i, class_id in enumerate(m_test.box.ap_class_index):
    det_summary[f'AP50_{m_test.names[int(class_id)]}'] = round(float(m_test.box.ap50[i]), 4)

df_det = pd.DataFrame([det_summary])
df_det.to_csv(f'{OUT_DIR}/metrics_detection.csv', index=False)
print('Đã lưu bảng kết quả Detection tại:', f'{OUT_DIR}/metrics_detection.csv')
display(df_det)

## 4. Huấn luyện Mô hình Ước lượng Tư thế 11 điểm thân trên (`yolo26s-pose.pt`)
Xác định khung `student` và 11 điểm khớp thân trên (`nose`, `left_eye`, `right_eye`, `left_ear`, `right_ear`, `left_shoulder`, `right_shoulder`, `left_elbow`, `right_elbow`, `left_wrist`, `right_wrist`). Kết quả lưu tại `/content/drive/MyDrive/ket_thuc_hoc_phan_AI/yolo26s_pose_11kpts`.

In [ ]:
pose_lbls = glob.glob(f'{POSE_DIR}/train/labels/*.txt')
if pose_lbls:
    best_pose, pose_h, pose_vram = train_with_backup(
        run_name='yolo26s_pose_11kpts',
        base_weights='yolo26s-pose.pt',
        data_yaml=DATA_POSE_YAML
    )
    mp_val  = best_pose.val(data=DATA_POSE_YAML, imgsz=IMGSZ, batch=BATCH, split='val',  project=OUT_DIR, name='yolo26s_pose_val',  exist_ok=True)
    mp_test = best_pose.val(data=DATA_POSE_YAML, imgsz=IMGSZ, batch=BATCH, split='test', project=OUT_DIR, name='yolo26s_pose_test', exist_ok=True)
    pose_summary = {
        'model': 'yolo26s_pose_11kpts',
        'val_box_mAP50': round(float(mp_val.box.map50), 4),
        'val_box_mAP50-95': round(float(mp_val.box.map), 4),
        'val_pose_mAP50': round(float(mp_val.pose.map50), 4),
        'val_pose_mAP50-95': round(float(mp_val.pose.map), 4),
        'box_mAP50': round(float(mp_test.box.map50), 4),
        'box_mAP50-95': round(float(mp_test.box.map), 4),
        'pose_mAP50': round(float(mp_test.pose.map50), 4),
        'pose_mAP50-95': round(float(mp_test.pose.map), 4),
        'pose_Precision': round(float(mp_test.pose.mp), 4),
        'pose_Recall': round(float(mp_test.pose.mr), 4),
        'infer_ms': round(float(mp_test.speed['inference']), 2),
        'vram_GB': pose_vram,
        'train_h': pose_h,
    }
    df_pose = pd.DataFrame([pose_summary])
    df_pose.to_csv(f'{OUT_DIR}/metrics_pose.csv', index=False)
    print('Đã lưu bảng kết quả Pose tại:', f'{OUT_DIR}/metrics_pose.csv')
    display(df_pose)
else:
    print('Lưu ý: Bản zip trên Drive chưa có thư mục labels_pose. Bạn hãy tải thêm file nhẹ labels_student_pose_11kpts.zip (3.5 MB) lên MyDrive để train mô hình Pose 11 điểm, hoặc dùng trực tiếp yolo26s-pose.pt ở bước 5!')

## 5. Chương trình Thử nghiệm: Kết hợp Nhận diện Vật thể + Tư thế 11 điểm & Cảnh báo Nghi vấn
Lưu ảnh kết quả cảnh báo tại `/content/drive/MyDrive/ket_thuc_hoc_phan_AI/demo_alerts/`.

In [ ]:
import os, glob, math, cv2, numpy as np
from google.colab import drive
from ultralytics import YOLO
from IPython.display import Image as IPyImage, display

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

OUT_DIR  = '/content/drive/MyDrive/ket_thuc_hoc_phan_AI'
DET_PT   = f'{OUT_DIR}/yolo26s_det_4classes/weights/best.pt'
POSE_PT  = f'{OUT_DIR}/yolo26s_pose_11kpts/weights/best.pt'

# Đường dẫn ảnh tùy ý trên Google Drive (mặc định dùng 00239.jpg hoặc lấy ảnh từ tập Test)
CUSTOM_IMG = '/content/drive/MyDrive/00239.jpg'
test_imgs  = sorted(glob.glob('/content/dataset_sv_vi_pham/det/test/images/*.jpg'))
IMG_PATH   = CUSTOM_IMG if os.path.exists(CUSTOM_IMG) else (test_imgs[0] if test_imgs else None)

det_model  = YOLO(DET_PT if os.path.exists(DET_PT) else 'yolo26s.pt')
pose_model = YOLO(POSE_PT if os.path.exists(POSE_PT) else 'yolo26s-pose.pt')

# Các cặp nối xương chỉ lấy 11 điểm thân trên (bỏ toàn bộ điểm thân dưới)
UPPER_SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),       # Mũi - Mắt - Tai
    (5, 6), (0, 5), (0, 6),               # Vai trái - Vai phải - Mũi
    (5, 7), (7, 9),                       # Tay trái (Vai -> Khuỷu -> Cổ tay)
    (6, 8), (8, 10)                       # Tay phải (Vai -> Khuỷu -> Cổ tay)
]

def analyze_exam_frame(img_path, conf_det=0.25, conf_pose=0.30):
    img = cv2.imread(img_path)
    res_det  = det_model.predict(img, imgsz=960, conf=conf_det, verbose=False)[0]
    res_pose = pose_model.predict(img, imgsz=960, conf=conf_pose, verbose=False)[0]

    phones, papers, calcs = [], [], []
    colors = {'student': (80, 200, 80), 'phone': (0, 0, 255), 'paper': (255, 180, 0), 'calculator': (255, 0, 180)}

    # 1. Vẽ khung nhận diện 4 lớp: student, phone, paper, calculator
    for box, cid, cf in zip(res_det.boxes.xyxy.cpu().numpy(), res_det.boxes.cls.cpu().numpy(), res_det.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = map(int, box)
        cname = res_det.names[int(cid)]
        if cname == 'person': cname = 'student'
        if cname == 'phone': phones.append((x1, y1, x2, y2))
        elif cname == 'paper': papers.append((x1, y1, x2, y2))
        elif cname == 'calculator': calcs.append((x1, y1, x2, y2))
        col = colors.get(cname, (80, 200, 80))
        cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
        cv2.putText(img, f'{cname} {cf:.2f}', (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

    alerts = []
    if phones:
        alerts.append(f'PHAT HIEN {len(phones)} DIEN THOAI TRONG PHONG THI!')

    # 2. Vẽ 11 điểm Pose thân trên (từ index 0 đến 10) & kiểm tra quy tắc tư thế nghi vấn
    if res_pose.keypoints is not None and len(res_pose.keypoints.data):
        for s_idx, (pbox, kpts) in enumerate(zip(res_pose.boxes.xyxy.cpu().numpy(), res_pose.keypoints.data.cpu().numpy())):
            kpts_upper = kpts[:11]  # Chỉ lấy đúng 11 điểm thân trên
            for i, j in UPPER_SKELETON:
                if i < len(kpts_upper) and j < len(kpts_upper) and kpts_upper[i][2] > 0.3 and kpts_upper[j][2] > 0.3:
                    pt1 = (int(kpts_upper[i][0]), int(kpts_upper[i][1]))
                    pt2 = (int(kpts_upper[j][0]), int(kpts_upper[j][1]))
                    cv2.line(img, pt1, pt2, (0, 255, 255), 2)
            for kx, ky, kc in kpts_upper:
                if kc > 0.3:
                    cv2.circle(img, (int(kx), int(ky)), 4, (0, 128, 255), -1)

            # Quy tắc A: Bàn tay (left_wrist=9, right_wrist=10) đặt gần điện thoại (phone)
            for w_idx in (9, 10):
                if w_idx < len(kpts_upper) and kpts_upper[w_idx][2] > 0.3:
                    wx, wy = kpts_upper[w_idx][0], kpts_upper[w_idx][1]
                    for px1, py1, px2, py2 in phones:
                        pcx, pcy = (px1 + px2) / 2, (py1 + py2) / 2
                        if math.hypot(wx - pcx, wy - pcy) < 120:
                            alerts.append(f'student#{s_idx+1}: TAY CAM/GAN DIEN THOAI')
                            cv2.putText(img, 'NGHI VAN: DUNG PHONE', (int(pbox[0]), int(pbox[1]) - 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            # Quy tắc B: Quay đầu nhìn ngang dựa vào mũi (0) và 2 vai (5, 6)
            if len(kpts_upper) >= 7 and kpts_upper[0][2] > 0.4 and kpts_upper[5][2] > 0.4 and kpts_upper[6][2] > 0.4:
                nx = kpts_upper[0][0]
                lsx, rsx = kpts_upper[5][0], kpts_upper[6][0]
                sh_w = abs(lsx - rsx)
                sh_mid = (lsx + rsx) / 2.0
                if sh_w > 25 and abs(nx - sh_mid) / sh_w > 0.42:
                    alerts.append(f'student#{s_idx+1}: QUAY DAU NGANG')
                    cv2.putText(img, 'CHU Y: QUAY DAU NGANG', (int(pbox[0]), int(pbox[1]) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 140, 255), 2)

    out_dir = f'{OUT_DIR}/demo_alerts'
    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/alert_{os.path.basename(img_path)}'
    cv2.imwrite(out_file, img)
    print(f'Đã phân tích: {img_path} -> Lưu kết quả tại: {out_file}')
    if alerts:
        print('⚠️ CẢNH BÁO PHÁT HIỆN:', ' | '.join(alerts))
    display(IPyImage(filename=out_file))

if IMG_PATH:
    analyze_exam_frame(IMG_PATH)

## 6. Đóng gói & Tải toàn bộ Kết quả trong `ket_thuc_hoc_phan_AI` về Máy tính

In [ ]:
import os, glob, zipfile
from google.colab import files

OUT_DIR = '/content/drive/MyDrive/ket_thuc_hoc_phan_AI'
report_zip = f'{OUT_DIR}/bao_cao_ket_qua_hoc_phan_AI.zip'
with zipfile.ZipFile(report_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in glob.glob(f'{OUT_DIR}/**/*', recursive=True):
        if os.path.isfile(fpath) and not fpath.endswith(('.pt', '.zip')):
            zf.write(fpath, os.path.relpath(fpath, OUT_DIR))
print('Đã lưu file nén báo cáo tại:', report_zip)
files.download(report_zip)